In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# architecture of encoder
encoder = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(8, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 8, kernel_size=3),
    nn.ReLU(),
    nn.Conv2d(8, 8, kernel_size=3),
    nn.Sigmoid(),
    nn.Flatten()
)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, encoder, num_features, num_classes, num_hidden=512):
        super().__init__()
        self.encoder = encoder
        self.perceptron = nn.Sequential(
            nn.Linear(num_features,num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden,num_classes),
        )
    def forward(self, x):
        features = self.encoder(x)
        logits = self.perceptron(features)
        if self.training:
            return logits
        else:
            return F.softmax(logits, dim=1)

In [ ]:
classifier = Classifier(encoder,72,10).to(device)

In [ ]:
!wget http://agentspace.org/download/mnist_classifier.pth
model_name = 'mnist_classifier.pth'
classifier.load_state_dict(torch.load(model_name, map_location=device))

In [ ]:
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
sample = test_dataset[0][0]
print(sample.shape)
plt.imshow(sample.squeeze(0), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
with torch.no_grad():
    logits = classifier(sample.unsqueeze(0).to(device)).squeeze(0).detach().cpu().numpy()

category = np.argmax(logits)
print(category)

In [ ]:
classifier

In [ ]:
# convert the perceptron into convolutional layers
fc1 = classifier.perceptron[0]  # Linear(72 → 512)
fc2 = classifier.perceptron[2]  # Linear(512 → 10)
conv_fc1 = nn.Conv2d(8, fc1.out_features, kernel_size=3, bias=True).to(device)
conv_fc2 = nn.Conv2d(fc1.out_features, fc2.out_features, kernel_size=1, bias=True).to(device)
# copy weights
with torch.no_grad():
    conv_fc1.weight.copy_(fc1.weight.view(fc1.out_features, 8, 3, 3))
    conv_fc1.bias.copy_(fc1.bias)
    conv_fc2.weight.copy_(fc2.weight.view(fc2.out_features, fc2.in_features, 1, 1))
    conv_fc2.bias.copy_(fc2.bias)
# create fully-convolutional perceptron
fully_convolutional_perceptron = nn.Sequential(
    conv_fc1,
    nn.ReLU(),
    conv_fc2
)
# remove flattening from the encoder
encoder_without_flattening = nn.Sequential(*list(classifier.encoder.children())[:-1])
# compose the new model
model = nn.Sequential(
    encoder_without_flattening,
    fully_convolutional_perceptron,
    nn.Softmax(dim=1)
)
model

In [ ]:
with torch.no_grad():
    logits = model(sample.unsqueeze(0).to(device)).squeeze(0).detach().cpu().numpy()

print(logits.shape)
print(logits)
category = np.argmax(logits[:,0,0])
print(category)

In [ ]:
# Save model and weights
def save():
    torch.save(model.state_dict(), model_name) # weights only
    print(f'Saved trained model at {model_name}')

In [ ]:
model_name = 'mnist_simple_detector.pth'
save()

In [ ]:
# download model
from google.colab import files
files.download(model_name)

In [ ]:
# show results
def render(digit):
    img = np.zeros((28, 28), dtype=np.uint8)
    cv2.putText(img, str(digit), (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
    return img

plt.figure(figsize=(20, 40))
for b in range(10):
    for i in range(10):
        input_img = (x_sample[10*b+i].squeeze(0).detach().numpy()*255).astype(np.uint8)
        plt.subplot(20, 10, 20*b+i+1)
        plt.imshow(input_img, cmap='gray')
        plt.axis('off')
        output_img = render(output_categories[10*b+i])
        plt.subplot(20, 10, 20*b+i+1+10)
        plt.imshow(~output_img, cmap='gray')
        plt.axis('off')

plt.show()